# Revisión de `team_priors_latest.parquet`

Ejecuta las celdas en orden para explorar el contenido del archivo. La revisión solo lee los datos.

In [18]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Encuentra el archivo desde la raíz del proyecto o desde review/.
ruta_relativa = Path("data/schedule_features_2016_2025.parquet")
ruta = next(
    (base / ruta_relativa for base in [Path.cwd(), *Path.cwd().parents]
     if (base / ruta_relativa).is_file()),
    None,
)
if ruta is None:
    raise FileNotFoundError(
        "No se encontró data/team_priors_latest.parquet. "
        "Abre el notebook desde el proyecto NFL-elo-model."
    )

df = pd.read_parquet(ruta)
print(f"Archivo: {ruta}")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
display(df.head(10))

Archivo: /workspaces/NFL-elo-model/data/schedule_features_2016_2025.parquet
Filas: 3,033 | Columnas: 24


,game_id,season,game_type,week,away_team,away_score,home_team,home_score,result,total,...,roof,surface,r_away,r_home,off_epa_pp_away,off_epa_pp_home,def_epa_pp_away,def_epa_pp_home,turnover_rate_away,turnover_rate_home
0,2016_01_BUF_BAL,2016,REG,1,BUF,7.0,BAL,13.0,6.0,20.0,...,outdoors,grass,1511.742063,1435.075595,0.053017,-0.069415,0.030410,0.013555,0.016427,0.024551
1,2016_01_CAR_DEN,2016,REG,1,CAR,20.0,DEN,21.0,1.0,41.0,...,outdoors,grass,1652.028309,1626.289453,0.074605,-0.099792,-0.135160,-0.176772,0.019624,0.027500
2,2016_01_CHI_HOU,2016,REG,1,CHI,14.0,HOU,23.0,9.0,37.0,...,closed,fieldturf,1446.552116,1506.358032,-0.022395,-0.069756,0.049487,-0.105550,0.019173,0.021739
3,2016_01_CIN_NYJ,2016,REG,1,CIN,23.0,NYJ,22.0,-1.0,45.0,...,outdoors,fieldturf,1571.835192,1545.809083,0.054443,0.031003,-0.049750,-0.111147,0.020251,0.021863
4,2016_01_CLE_PHI,2016,REG,1,CLE,10.0,PHI,29.0,19.0,39.0,...,outdoors,grass,1389.654847,1473.103932,-0.077269,-0.047730,0.085443,0.042589,0.027300,0.028169
5,2016_01_DET_IND,2016,REG,1,DET,39.0,IND,35.0,-4.0,74.0,...,open,fieldturf,1484.109265,1467.315771,-0.013646,-0.100794,0.032510,0.002720,0.022312,0.026601
6,2016_01_GB_JAX,2016,REG,1,GB,27.0,JAX,23.0,-4.0,50.0,...,outdoors,grass,1568.247508,1410.447243,-0.016822,-0.065259,-0.048985,0.035399,0.016623,0.027055
7,2016_01_LA_SF,2016,REG,1,LA,0.0,SF,28.0,28.0,28.0,...,outdoors,grass,1472.654161,1401.824930,-0.158323,-0.087286,-0.066548,0.064800,0.022173,0.016376
8,2016_01_MIA_SEA,2016,REG,1,MIA,10.0,SEA,12.0,2.0,22.0,...,outdoors,fieldturf,1427.189268,1597.813725,-0.034968,0.078317,0.047376,-0.075868,0.019313,0.016319
9,2016_01_MIN_TEN,2016,REG,1,MIN,25.0,TEN,16.0,-9.0,41.0,...,outdoors,grass,1568.038919,1380.681100,-0.014017,-0.123630,-0.055190,0.034322,0.017329,0.033623


In [62]:
df[df.game_id == "2025_22_SEA_NE"]

,game_id,season,game_type,week,away_team,away_score,home_team,home_score,result,total,...,roof,surface,r_away,r_home,off_epa_pp_away,off_epa_pp_home,def_epa_pp_away,def_epa_pp_home,turnover_rate_away,turnover_rate_home
2760,2025_22_SEA_NE,2025,SB,22,SEA,29.0,NE,13.0,-16.0,42.0,...,outdoors,grass,1748.746739,1665.011062,0.109102,-0.022013,-0.128782,-0.266123,0.010315,0.018676


In [60]:
df[
    ((df.away_team == "NE") | (df.home_team == "NE"))
    & (df.season == 2026)
    & (df.week.isin([1, 2]))
]

,game_id,season,game_type,week,away_team,away_score,home_team,home_score,result,total,...,roof,surface,r_away,r_home,off_epa_pp_away,off_epa_pp_home,def_epa_pp_away,def_epa_pp_home,turnover_rate_away,turnover_rate_home
2771,2026_01_NE_SEA,2026,REG,1,NE,NaN,SEA,NaN,NaN,NaN,...,outdoors,fieldturf,1599.633090,1710.68526,-0.110981,0.064463,-0.205700,-0.187854,0.027217,0.007427
2790,2026_02_PIT_NE,2026,REG,2,PIT,NaN,NE,NaN,NaN,NaN,...,outdoors,fieldturf,1510.166696,1632.84412,-0.136385,-0.110981,0.032267,-0.205700,0.016266,0.027217


## Columnas y calidad de los datos

Tipos de datos, valores faltantes, valores distintos y filas duplicadas.

In [15]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "valores_distintos": df.nunique(),
})
resumen.index.name = "columna"
display(resumen)
print(f"Filas completamente duplicadas: {df.duplicated().sum()}")
print(f"Equipos distintos: {df["team"].nunique()}")
display(df[["season", "week"]].drop_duplicates().sort_values(["season", "week"]))

,tipo,nulos,valores_distintos
columna,,,
team,object,0,32
season,int64,0,1
week,int64,0,5
as_of_game_id,object,0,27
r_elo,float64,0,32
off_epa_pp,float64,0,32
def_epa_allowed_pp,float64,0,32
turnover_rate,float64,0,32


Filas completamente duplicadas: 0
Equipos distintos: 32


columna,season,week
0,2025,18
4,2025,19
3,2025,20
9,2025,21
21,2025,22


## Todos los registros

Tabla completa ordenada por equipo.

In [16]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df.sort_values("team").reset_index(drop=True))

columna,team,season,week,as_of_game_id,r_elo,off_epa_pp,def_epa_allowed_pp,turnover_rate
0,ARI,2025,18,2025_18_ARI_LA,1324.244651,-0.010251,0.193810,0.021939
1,ATL,2025,18,2025_18_NO_ATL,1443.645736,-0.037490,-0.015972,0.016010
2,BAL,2025,18,2025_18_BAL_PIT,1583.606597,0.144231,0.025904,0.022719
3,BUF,2025,20,2025_20_BUF_DEN,1681.426767,0.128170,-0.044853,0.024096
4,CAR,2025,19,2025_19_LA_CAR,1367.255703,-0.092139,0.045208,0.023825
5,CHI,2025,20,2025_20_LA_CHI,1532.306334,0.065272,0.091479,0.020370
6,CIN,2025,18,2025_18_CLE_CIN,1468.700893,0.103357,0.005215,0.018543
7,CLE,2025,18,2025_18_CLE_CIN,1362.351251,-0.194643,-0.065014,0.032048
8,DAL,2025,18,2025_18_DAL_NYG,1430.741832,0.044771,0.225547,0.020375
9,DEN,2025,21,2025_21_NE_DEN,1665.214379,-0.094310,-0.113857,0.019237


## Estadísticas y ranking Elo

In [4]:
display(df.describe(include="all").T)
display(df.sort_values("r_elo", ascending=False).reset_index(drop=True))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
columna,,,,,,,,,,,
team,32,32,ARI,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
season,32.0,NaN,NaN,NaN,2025.0,0.0,2025.0,2025.0,2025.0,2025.0,2025.0
week,32.0,NaN,NaN,NaN,18.875,1.237844,18.0,18.0,18.0,19.25,22.0
as_of_game_id,32,27,2025_18_NO_ATL,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_elo,32.0,NaN,NaN,NaN,1499.996918,135.609608,1249.573035,1418.933571,1512.039661,1591.023443,1780.913681
off_epa_pp,32.0,NaN,NaN,NaN,-0.042367,0.121785,-0.311453,-0.125701,-0.030157,0.051377,0.166119
def_epa_allowed_pp,32.0,NaN,NaN,NaN,-0.010493,0.135941,-0.292709,-0.10106,0.004389,0.090177,0.242668
turnover_rate,32.0,NaN,NaN,NaN,0.020812,0.006532,0.007427,0.017974,0.021495,0.023893,0.034095


columna,team,season,week,as_of_game_id,r_elo,off_epa_pp,def_epa_allowed_pp,turnover_rate
0,SEA,2025,22,2025_22_SEA_NE,1780.913681,0.064463,-0.187875,0.007427
1,BUF,2025,20,2025_20_BUF_DEN,1681.426767,0.128170,-0.044853,0.024096
2,LA,2025,21,2025_21_LA_SEA,1676.802236,0.166119,0.058131,0.008169
3,DEN,2025,21,2025_21_NE_DEN,1665.214379,-0.094310,-0.113857,0.019237
4,HOU,2025,20,2025_20_HOU_NE,1661.851339,-0.129277,-0.253838,0.034095
5,NE,2025,22,2025_22_SEA_NE,1632.844120,-0.111594,-0.205700,0.027217
6,PHI,2025,19,2025_19_SF_PHI,1618.731874,-0.018505,-0.074668,0.009270
7,DET,2025,18,2025_18_DET_CHI,1601.125523,0.004920,0.003563,0.026555
8,SF,2025,20,2025_20_SF_SEA,1587.656083,-0.063207,0.102412,0.031307
9,BAL,2025,18,2025_18_BAL_PIT,1583.606597,0.144231,0.023622,0.022719


## Consultar un equipo

Cambia la abreviatura para revisar otro equipo.

In [ ]:
equipo = "KC"
display(df.loc[df["team"].eq(equipo.upper())])

## Descargar el calendario de 2026: semana 1

Esta sección requiere conexión a internet. Descarga el calendario de 2026 y guarda los partidos de la primera semana de temporada regular en `data/schedule_2026_week1.parquet`. Si el archivo ya existe, se reemplaza.

In [5]:
from pathlib import Path

import nflreadpy as nfl
from IPython.display import display

# Localiza la raíz desde el proyecto o desde review/.
raiz_proyecto = next(
    (base for base in [Path.cwd(), *Path.cwd().parents]
     if (base / "pyproject.toml").is_file() and (base / "data").is_dir()),
    None,
)
if raiz_proyecto is None:
    raise FileNotFoundError("Abre el notebook desde el proyecto NFL-elo-model.")

schedule = nfl.load_schedules(seasons=2026).to_pandas()
print(f"Partidos en el calendario de 2026: {len(schedule)}")

Partidos en el calendario de 2026: 272


In [13]:
schedule.iloc[0]

game_id             2026_01_NE_SEA
season                        2026
game_type                      REG
week                             1
gameday                 2026-09-09
weekday                  Wednesday
gametime                     20:20
away_team                       NE
away_score                     NaN
home_team                      SEA
home_score                     NaN
location                      Home
result                         NaN
total                          NaN
overtime                       NaN
old_game_id             2026090900
gsis                           NaN
nfl_detail_id                 None
pfr                   202609090sea
pff                            NaN
espn                     401872656
ftn                            NaN
away_rest                        7
home_rest                        7
away_moneyline               142.0
home_moneyline              -170.0
spread_line                    3.0
away_spread_odds            -102.0
home_spread_odds    

In [ ]:
week1 = schedule.loc[
    (schedule["game_type"] == "REG")
    & (schedule["week"] == 1)
].copy()

week1 = week1.sort_values(["gameday", "gametime", "game_id"])

ruta_salida = raiz_proyecto / "data" / "schedule_2026_week1.parquet"
week1.to_parquet(ruta_salida, index=False)

print(f"Archivo guardado: {ruta_salida}")
print(f"Partidos de la semana 1: {len(week1)}")
display(week1)

In [17]:
df_lts = nfl.load_team_stats(seasons=2026).to_pandas()

ConnectionError: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet

In [1]:
from pathlib import Path
import pandas as pd

# Ruta absoluta para poder ejecutar la celda desde cualquier directorio.
ruta_priors = Path("/workspaces/NFL-elo-model/data/team_priors_latest.parquet")
team_priors = pd.read_parquet(ruta_priors)

print(f"Filas: {team_priors.shape[0]} | Columnas: {team_priors.shape[1]}")
team_priors.head(10)


Filas: 32 | Columnas: 8


,team,season,week,as_of_game_id,r_elo,off_epa_pp,def_epa_allowed_pp,turnover_rate
0,ARI,2026,5,2026_05_DET_ARI,1348.085876,-0.062562,0.168352,0.021822
1,ATL,2026,5,2026_05_BAL_ATL,1458.066239,0.069826,0.016212,0.020059
2,BAL,2026,5,2026_05_BAL_ATL,1590.735350,0.149226,0.011609,0.010224
3,BUF,2026,5,2026_05_BUF_LA,1672.674562,0.197443,0.128004,0.036488
4,CAR,2026,4,2026_04_DET_CAR,1434.553878,0.082347,0.039327,0.010661
5,CHI,2026,5,2026_05_CHI_GB,1535.268434,-0.040823,0.004185,0.034464
6,CIN,2026,5,2026_05_CIN_MIA,1519.519022,0.114158,0.030215,0.018226
7,CLE,2026,5,2026_05_CLE_NYJ,1413.168828,-0.104821,-0.057942,0.025489
8,DAL,2026,5,2026_05_TB_DAL,1434.492268,0.099462,0.193546,0.016471
9,DEN,2026,5,2026_05_DEN_LAC,1612.959630,-0.086912,-0.071204,0.015272


In [1]:
from pathlib import Path
import pandas as pd

# Ruta absoluta para poder ejecutar la celda desde cualquier directorio.
ruta_priors = Path("/workspaces/NFL-elo-model/data/team_priors_latest.parquet")
team_priors = pd.read_parquet(ruta_priors)

print(f"Filas: {team_priors.shape[0]} | Columnas: {team_priors.shape[1]}")
team_priors.head(10)


Filas: 32 | Columnas: 8


,team,season,week,as_of_game_id,r_elo,off_epa_pp,def_epa_allowed_pp,turnover_rate
0,ARI,2026,6,2026_06_ARI_LA,1366.920126,-0.050814,0.095909,0.019712
1,ATL,2026,6,2026_06_CHI_ATL,1436.865358,-0.032563,0.001564,0.020665
2,BAL,2026,6,2026_06_BAL_CLE,1572.531746,0.079745,-0.027482,0.012547
3,BUF,2026,6,2026_06_BUF_LV,1678.912426,0.189262,0.118782,0.026271
4,CAR,2026,6,2026_06_CAR_PHI,1470.939439,0.071490,-0.022095,0.007676
5,CHI,2026,6,2026_06_CHI_ATL,1556.469316,-0.039502,-0.079824,0.028872
6,CIN,2026,5,2026_05_CIN_MIA,1519.519022,0.114158,0.030215,0.018226
7,CLE,2026,6,2026_06_BAL_CLE,1431.372432,-0.111310,-0.069416,0.018352
8,DAL,2026,6,2026_06_DAL_GB,1420.756360,0.033421,0.148961,0.019975
9,DEN,2026,6,2026_06_SEA_DEN,1627.667797,-0.103568,-0.077216,0.010996
